# Advanced Enterprise RAG System for Domain-Specific Question Answering

## Course

**Conversational AI**

## Assignment

**Advanced Retrieval-Augmented Generation (RAG) Architecture Design and Evaluation**

## Domain

**Enterprise Healthcare Knowledge Support System**

## Dataset

**MedQuAD (Medical Question Answering Dataset)**

## Student Details

* **Name:** <Your Name>
* **Roll Number:** <Your Roll Number>
* **Program:** M.Tech AIML
* **Institution:** BITS Pilani WILP
* **Date:** 08 August 2026


## 2. Problem Statement

An enterprise healthcare organization maintains a large collection of medical knowledge articles, treatment guidelines, symptom descriptions, diagnostic procedures, and frequently asked questions. Users require a conversational question-answering system capable of retrieving relevant information from this domain-specific knowledge base and generating grounded, explainable responses.

The objective of this assignment is to design and implement an **advanced Retrieval-Augmented Generation (RAG) architecture** that supports:

* intelligent document ingestion and chunking,
* metadata-aware retrieval,
* hybrid sparse and dense retrieval,
* cross-encoder re-ranking,
* contextual evidence assembly,
* agentic query routing and reformulation,
* and systematic evaluation of retrieval and generation quality.

The implementation focuses on building a **domain-specific healthcare knowledge support assistant** using the **MedQuAD dataset** while ensuring that the system remains lightweight, reproducible, and executable in a CPU-based Jupyter Notebook environment.


## 3. Dataset Description and Justification

This assignment uses the **MedQuAD (Medical Question Answering Dataset)** obtained from Hugging Face. MedQuAD is derived from authoritative **National Institutes of Health (NIH)** medical information repositories and contains structured question-answer pairs covering diseases, symptoms, diagnosis, treatment, prevention, medications, and healthcare guidance.

### Why MedQuAD was Selected

The dataset is suitable for an **enterprise healthcare knowledge-management scenario** because it resembles a real-world organizational repository containing:

* medical knowledge articles,
* patient-support FAQs,
* treatment and prevention guidance,
* disease-specific informational documents,
* and operational healthcare knowledge resources.

### Advantages for Advanced RAG

* Clean and structured textual content
* Rich domain-specific terminology
* Natural question and answer format
* Suitable for semantic and keyword retrieval
* Lightweight enough for CPU execution
* Supports chunking, metadata enrichment, re-ranking, and evaluation tasks

The dataset will be used throughout all subsequent tasks, ensuring a **consistent end-to-end implementation without changing the corpus during the assignment lifecycle**.


## 4. Environment Setup

The following libraries are required for implementing the advanced RAG pipeline. The installation cell needs to be executed only once in the Virtual Lab environment.


In [28]:
# Run this cell only once if the libraries are not already installed

!pip install datasets
!pip install sentence-transformers
!pip install faiss-cpu
!pip install rank-bm25
!pip install pandas pyarrow scikit-learn
!pip install matplotlib seaborn
!pip install tf-keras

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


## 5. Dataset Download and Caching

To avoid repeated downloads during experimentation, the MedQuAD dataset is downloaded once and stored locally as a **Parquet file**. Subsequent notebook executions will load the cached file directly from disk, making the workflow faster and more reproducible.


## 5.1 — Download and Cache Dataset

In [29]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd

# Create data directory
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Local cache file
DATASET_FILE = DATA_DIR / "medquad.parquet"

if DATASET_FILE.exists():
    print("Loading cached MedQuAD dataset...")
    df = pd.read_parquet(DATASET_FILE)
else:
    print("Downloading MedQuAD dataset from Hugging Face...")

    dataset = load_dataset("lavita/MedQuAD", split="train")

    # Convert to pandas DataFrame
    df = dataset.to_pandas()

    # Save locally for future runs
    df.to_parquet(DATASET_FILE, index=False)

    print(f"Dataset downloaded and cached at: {DATASET_FILE}")

print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")

Loading cached MedQuAD dataset...
Total records: 47,441
Total columns: 13


## 5.2 Inspect Dataset Structure

In [30]:
# Display dataset structure
print("Dataset Columns:")
print(df.columns.tolist())

print("\\nFirst 3 Records:")
display(df.head(3))

Dataset Columns:
['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']
\nFirst 3 Records:


,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,NaN,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,NaN,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,NaN,C0343073,T047,Disorders,KWWH,0000559-3,keratoderma with woolly hair,genetic changes,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."


## 5.3 Dataset Statistics

In [31]:
# Basic dataset statistics
stats = {
    "Total Records": len(df),
    "Total Columns": len(df.columns),
    "Missing Questions": df["question"].isna().sum() if "question" in df.columns else "N/A",
    "Missing Answers": df["answer"].isna().sum() if "answer" in df.columns else "N/A"
}

stats_df = pd.DataFrame(stats.items(), columns=["Metric", "Value"])
display(stats_df)

,Metric,Value
0,Total Records,47441
1,Total Columns,13
2,Missing Questions,0
3,Missing Answers,31034


## 5.4 Markdown Analysis

### Observation and Analysis

The MedQuAD dataset provides a structured collection of medical question-answer pairs that can be treated as **domain-specific enterprise knowledge documents**. Each record contains a user-oriented medical question and an authoritative answer derived from NIH information repositories.

Key observations from the dataset inspection include:

* The corpus is already available in **clean textual form**, eliminating the need for complex PDF extraction and OCR processing.
* Questions and answers can be combined into a **unified document representation** for retrieval experiments.
* The dataset is sufficiently large to demonstrate **chunking, hybrid retrieval, re-ranking, and evaluation techniques** while remaining lightweight enough for CPU-based execution.
* The presence of structured fields enables the creation of **metadata-aware retrieval pipelines** in later tasks.

This confirms that MedQuAD is an appropriate and stable corpus for implementing the complete Advanced RAG assignment workflow.


## 6. Model Loading and Local Caching

The Advanced RAG pipeline requires two lightweight transformer models:

1. **SentenceTransformer** for generating dense semantic embeddings used in FAISS retrieval.
2. **CrossEncoder** for re-ranking retrieved candidate chunks based on query-document relevance.

To avoid repeated downloads during notebook reruns, both models are stored locally inside a `models/` directory and loaded from disk whenever available.


## 6.1 Create Model Cache Directories

In [32]:
from pathlib import Path

# Create local model cache directory
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

EMBEDDING_MODEL_DIR = MODEL_DIR / "all-MiniLM-L6-v2"
RERANKER_MODEL_DIR = MODEL_DIR / "ms-marco-MiniLM-L-6-v2"

print(f"Model cache directory: {MODEL_DIR.resolve()}")

Model cache directory: C:\MTech\Bits\sem3\Conversational_AI\assignment2\assignmentdoc\advanced-rag-enterprise\Advanced RAG assignment\models


## 6.2 Load Embedding Model

In [33]:
from sentence_transformers import SentenceTransformer

if EMBEDDING_MODEL_DIR.exists():
    print("Loading cached embedding model...")
    embedding_model = SentenceTransformer(str(EMBEDDING_MODEL_DIR))
else:
    print("Downloading embedding model...")
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embedding_model.save(str(EMBEDDING_MODEL_DIR))
    print("Embedding model cached locally.")

print("Embedding model ready.")

Loading cached embedding model...
Embedding model ready.


## 6.3 Load Cross-Encoder Reranker

In [34]:
from sentence_transformers import CrossEncoder

if RERANKER_MODEL_DIR.exists():
    print("Loading cached reranker model...")
    reranker_model = CrossEncoder(str(RERANKER_MODEL_DIR))
else:
    print("Downloading reranker model...")
    reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    reranker_model.save(str(RERANKER_MODEL_DIR))
    print("Reranker model cached locally.")

print("Cross-encoder reranker ready.")

Loading cached reranker model...
Cross-encoder reranker ready.


## 6.4 Verify Models

In [35]:
# Verify that both models are loaded correctly
print("Embedding model:", type(embedding_model).__name__)
print("Reranker model:", type(reranker_model).__name__)

Embedding model: SentenceTransformer
Reranker model: CrossEncoder


## 6.5 Markdown Analysis

### Observation and Analysis

The notebook now uses two lightweight transformer models that are suitable for **CPU-based Advanced RAG experimentation**.

* **all-MiniLM-L6-v2** generates compact semantic embeddings that provide efficient dense retrieval while maintaining good semantic similarity performance.
* **ms-marco-MiniLM-L-6-v2** acts as a cross-encoder reranker that scores query-document pairs more accurately than vector similarity alone.

By caching the models locally, subsequent notebook executions avoid repeated downloads from Hugging Face, which significantly reduces startup time and improves reproducibility. This approach provides a practical balance between **enterprise-style efficiency** and **single-notebook simplicity**, which is appropriate for the assignment environment.


## 7. Corpus Preparation and Metadata Construction

Before implementing chunking and retrieval, the MedQuAD dataset must be transformed into a **document-oriented enterprise corpus**. The original dataset contains separate question and answer fields, which are combined into a unified document representation suitable for retrieval-augmented generation.

This section performs:

* text cleaning and normalization,
* document construction,
* metadata generation,
* unique document identifier assignment,
* and local persistence of the processed corpus for reuse throughout the notebook.


## 7.1 Select Relevant Columns

In [36]:
# Keep only useful columns if they exist
candidate_columns = [
    "question",
    "answer",
    "source",
    "focus_area",
    "url"
]

available_columns = [c for c in candidate_columns if c in df.columns]

corpus_df = df[available_columns].copy()

print("Selected columns:")
print(corpus_df.columns.tolist())

print(f"Records retained: {len(corpus_df):,}")

Selected columns:
['question', 'answer']
Records retained: 47,441


## 7.2 Clean and Normalize Text

In [37]:
import re

def clean_text(text):
    """Basic text cleaning for RAG processing."""

    text = str(text)

    # Remove extra whitespace
    text = re.sub(r"\\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text

corpus_df["question_clean"] = corpus_df["question"].apply(clean_text)
corpus_df["answer_clean"] = corpus_df["answer"].apply(clean_text)

print("Text cleaning completed.")

Text cleaning completed.


## 7.3 Create Unified Document Representation

In [38]:
# Create a unified document representation for retrieval
corpus_df["document_text"] = (
    "Medical Question: " + corpus_df["question_clean"] +
    "\\n\\nAuthoritative Answer: " + corpus_df["answer_clean"]
)

# Create unique document identifiers
corpus_df["document_id"] = [
    f"MEDQ_{i:06d}" for i in range(len(corpus_df))
]

display(corpus_df[[
    "document_id",
    "question_clean",
    "document_text"
]].head(2))

,document_id,question_clean,document_text
0,MEDQ_000000,What is (are) keratoderma with woolly hair ?,Medical Question: What is (are) keratoderma wi...
1,MEDQ_000001,How many people are affected by keratoderma wi...,Medical Question: How many people are affected...


## 7.4 Create Enterprise-Style Metadata

In [39]:
# Create enterprise-style metadata fields
corpus_df["document_type"] = "Healthcare Knowledge Article"
corpus_df["domain"] = "Healthcare Support"
corpus_df["knowledge_source"] = corpus_df.get("source", "NIH Medical Repository")
corpus_df["access_level"] = "Internal Healthcare Knowledge Base"

metadata_columns = [
    "document_id",
    "document_type",
    "domain",
    "knowledge_source",
    "access_level"
]

display(corpus_df[metadata_columns].head(3))

,document_id,document_type,domain,knowledge_source,access_level
0,MEDQ_000000,Healthcare Knowledge Article,Healthcare Support,NIH Medical Repository,Internal Healthcare Knowledge Base
1,MEDQ_000001,Healthcare Knowledge Article,Healthcare Support,NIH Medical Repository,Internal Healthcare Knowledge Base
2,MEDQ_000002,Healthcare Knowledge Article,Healthcare Support,NIH Medical Repository,Internal Healthcare Knowledge Base


## 7.5 Save Processed Corpus

In [40]:
from pathlib import Path

# Create processed data directory
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_FILE = PROCESSED_DIR / "medquad_processed.parquet"

corpus_df.to_parquet(PROCESSED_FILE, index=False)

print(f"Processed corpus saved to: {PROCESSED_FILE}")
print(f"Total processed documents: {len(corpus_df):,}")

Processed corpus saved to: data\processed\medquad_processed.parquet
Total processed documents: 47,441


## 7.6 Corpus Statistics

In [41]:
# Compute corpus statistics
corpus_stats = {
    "Total Documents": len(corpus_df),
    "Average Question Length": round(corpus_df["question_clean"].str.len().mean(), 2),
    "Average Answer Length": round(corpus_df["answer_clean"].str.len().mean(), 2),
    "Average Document Length": round(corpus_df["document_text"].str.len().mean(), 2)
}

stats_table = pd.DataFrame(corpus_stats.items(), columns=["Metric", "Value"])
display(stats_table)

,Metric,Value
0,Total Documents,47441.00
1,Average Question Length,51.54
2,Average Answer Length,452.75
3,Average Document Length,548.29


## 7.7 Markdown Analysis

### Observation and Analysis

The MedQuAD dataset has now been transformed into a **document-oriented enterprise healthcare corpus** suitable for advanced RAG experimentation. Each record is represented as a unified document containing a medical question and its authoritative answer, which more closely resembles a real-world knowledge-base article than the original QA-pair structure.

The metadata construction step introduces enterprise-style attributes such as **document identifiers, document type, domain, knowledge source, and access level**, enabling metadata-aware retrieval and contextual filtering in later tasks.

The processed corpus is persisted locally as a **Parquet file**, ensuring that subsequent notebook sections can reuse the cleaned documents without repeating preprocessing operations. This improves reproducibility, reduces execution time, and establishes a stable foundation for chunking, indexing, retrieval, and evaluation tasks.


## 8. Task 1 — Fixed-Size and Sliding-Window Chunking

Chunking is a critical preprocessing step in Retrieval-Augmented Generation systems because large documents cannot be embedded or retrieved efficiently as a single unit. The objective of chunking is to divide documents into smaller retrieval units while preserving enough contextual information to support accurate question answering.

This section implements two foundational chunking strategies:

1. **Fixed-size chunking** — splits documents into equal-sized character segments.
2. **Sliding-window chunking** — creates overlapping chunks so that contextual information is preserved across chunk boundaries.

The resulting chunks will be used later for hybrid retrieval, re-ranking, and contextual evidence assembly.


## 8.1 Select Sample Documents

In [42]:
# Use a small sample for chunking demonstration
sample_docs = corpus_df[[
    "document_id",
    "document_text",
    "question_clean"
]].head(5).copy()

print(f"Sample documents selected: {len(sample_docs)}")

display(sample_docs[["document_id", "question_clean"]].head())

Sample documents selected: 5


,document_id,question_clean
0,MEDQ_000000,What is (are) keratoderma with woolly hair ?
1,MEDQ_000001,How many people are affected by keratoderma wi...
2,MEDQ_000002,What are the genetic changes related to kerato...
3,MEDQ_000003,Is keratoderma with woolly hair inherited ?
4,MEDQ_000004,What are the treatments for keratoderma with w...


## 8.2 Fixed-Size Chunking Function

In [43]:
def fixed_size_chunk(text, chunk_size=300):
    """
    Split text into fixed-size character chunks.
    """

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks

# Demonstrate on the first document
sample_text = sample_docs.iloc[0]["document_text"]

fixed_chunks = fixed_size_chunk(sample_text, chunk_size=300)

print(f"Fixed-size chunks created: {len(fixed_chunks)}")

for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"\\n--- Fixed Chunk {i+1} ---")
    print(chunk[:250])

Fixed-size chunks created: 7
\n--- Fixed Chunk 1 ---
Medical Question: What is (are) keratoderma with woolly hair ?\n\nAuthoritative Answer: Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening he
\n--- Fixed Chunk 2 ---
ir that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmopl
\n--- Fixed Chunk 3 ---
 the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlik


## 8.3 Sliding-Window Chunking Function

In [44]:
def sliding_window_chunk(text, chunk_size=300, overlap=100):
    """
    Split text into overlapping chunks using a sliding window.
    """

    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]

        if chunk:
            chunks.append(chunk)

        if i + chunk_size >= len(text):
            break

    return chunks

sliding_chunks = sliding_window_chunk(
    sample_text,
    chunk_size=300,
    overlap=100
)

print(f"Sliding-window chunks created: {len(sliding_chunks)}")

for i, chunk in enumerate(sliding_chunks[:3]):
    print(f"\\n--- Sliding Chunk {i+1} ---")
    print(chunk[:250])

Sliding-window chunks created: 10
\n--- Sliding Chunk 1 ---
Medical Question: What is (are) keratoderma with woolly hair ?\n\nAuthoritative Answer: Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening he
\n--- Sliding Chunk 2 ---
crease the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only sca
\n--- Sliding Chunk 3 ---
The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet 


## 8.4 Semantic Chunking

In [45]:
import textwrap

def semantic_chunk(text, max_chunk_length=350):
    """
    Simple semantic chunking based on sentence boundaries.
    Groups related sentences together until the maximum chunk length is reached.
    """

    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\\s+', text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        # If adding the sentence exceeds the limit, finalize current chunk
        if len(current_chunk) + len(sentence) > max_chunk_length and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += " " + sentence

    # Add the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

# Demonstrate semantic chunking
semantic_chunks = semantic_chunk(sample_text, max_chunk_length=350)

print(f"Semantic chunks created: {len(semantic_chunks)}")

for i, chunk in enumerate(semantic_chunks[:3]):
    print(f"\\n--- Semantic Chunk {i+1} ---")
    print(textwrap.shorten(chunk, width=250, placeholder="..."))

Semantic chunks created: 1
\n--- Semantic Chunk 1 ---
Medical Question: What is (are) keratoderma with woolly hair ?\n\nAuthoritative Answer: Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening...


## 8.5 Compare Chunk Counts

In [46]:
# Compare all three chunking strategies
comparison_df = pd.DataFrame({
    "Strategy": ["Fixed-Size", "Sliding-Window", "Semantic"],
    "Chunk Size": [300, 300, 350],
    "Overlap": [0, 100, "Sentence-based"],
    "Number of Chunks": [
        len(fixed_chunks),
        len(sliding_chunks),
        len(semantic_chunks)
    ]
})

display(comparison_df)

,Strategy,Chunk Size,Overlap,Number of Chunks
0,Fixed-Size,300,0,7
1,Sliding-Window,300,100,10
2,Semantic,350,Sentence-based,1


## 8.6 Apply Chunking to the Full Corpus Sample

In [47]:
# Create chunk records using SEMANTIC chunking as the primary strategy
chunk_records = []

for _, row in sample_docs.iterrows():
    chunks = semantic_chunk(
        row["document_text"],
        max_chunk_length=350
    )

    for idx, chunk in enumerate(chunks):
        chunk_records.append({
            "document_id": row["document_id"],
            "chunk_id": f"{row['document_id']}_CHUNK_{idx:03d}",
            "chunk_text": chunk,
            "chunk_length": len(chunk),
            "chunk_strategy": "semantic"
        })

chunks_df = pd.DataFrame(chunk_records)

print(f"Total semantic chunks generated: {len(chunks_df)}")

display(chunks_df.head(5))

Total semantic chunks generated: 5


,document_id,chunk_id,chunk_text,chunk_length,chunk_strategy
0,MEDQ_000000,MEDQ_000000_CHUNK_000,Medical Question: What is (are) keratoderma wi...,2011,semantic
1,MEDQ_000001,MEDQ_000001_CHUNK_000,Medical Question: How many people are affected...,577,semantic
2,MEDQ_000002,MEDQ_000002_CHUNK_000,Medical Question: What are the genetic changes...,1954,semantic
3,MEDQ_000003,MEDQ_000003_CHUNK_000,Medical Question: Is keratoderma with woolly h...,420,semantic
4,MEDQ_000004,MEDQ_000004_CHUNK_000,Medical Question: What are the treatments for ...,1019,semantic


## 8.7 Save Chunked Corpus

In [48]:
from pathlib import Path

# Save chunked corpus for later retrieval tasks
CHUNK_DIR = Path("data/chunks")
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_FILE = CHUNK_DIR / "medquad_chunks.parquet"

chunks_df.to_parquet(CHUNK_FILE, index=False)

print(f"Chunked corpus saved to: {CHUNK_FILE}")

Chunked corpus saved to: data\chunks\medquad_chunks.parquet


## 8.8 Markdown Comparison Analysis

### Comparison of Chunking Strategies

Three chunking strategies were implemented and compared to understand their impact on retrieval quality in a RAG system.

#### 1. Fixed-Size Chunking

* Simple and computationally efficient
* Produces uniformly sized chunks
* Easy to index and store
* May split sentences or related concepts across chunk boundaries
* Can reduce retrieval quality when important context is divided between adjacent chunks

#### 2. Sliding-Window Chunking

* Preserves contextual continuity through overlapping regions
* Improves the probability that complete semantic information appears in at least one chunk
* Generally provides better retrieval performance for question-answering tasks
* Produces a larger number of chunks, increasing storage and indexing overhead

#### 3. Semantic Chunking

* Splits documents at **sentence boundaries** rather than arbitrary character positions
* Preserves semantic coherence and readability
* Produces chunks that contain complete ideas or explanations
* Particularly effective for healthcare knowledge articles where symptoms, diagnosis, and treatment information are expressed as connected sentences
* Slightly more computationally expensive than fixed-size chunking but usually produces higher-quality retrieval units

### Comparative Observation

| Strategy       | Context Preservation | Computational Cost | Retrieval Quality |
| -------------- | -------------------- | ------------------ | ----------------- |
| Fixed-Size     | Low                  | Low                | Moderate          |
| Sliding-Window | High                 | Medium             | High              |
| Semantic       | Very High            | Medium             | Very High         |

For healthcare knowledge documents, **semantic chunking provides the best balance between semantic coherence and retrieval effectiveness**, while **sliding-window chunking remains a strong practical alternative** for large-scale indexing scenarios.


## 8.9 Assignment Inference

### Inference for Task 1

The comparison shows that **semantic chunking produces the most coherent retrieval units** because chunks are created using sentence boundaries rather than arbitrary character positions. This preserves complete medical concepts, symptom descriptions, and treatment explanations, which is highly beneficial for healthcare question-answering systems.

However, semantic chunking may generate **variable-sized chunks**, which can introduce challenges for indexing efficiency, embedding consistency, and retrieval latency in very large enterprise corpora. Sliding-window chunking, while slightly less semantically precise, provides **more predictable chunk sizes and stable indexing behavior**.

For this assignment, **semantic chunking will be selected as the primary chunking strategy** because the corpus is moderate in size and the objective is to maximize **retrieval quality and contextual coherence** rather than optimize large-scale production throughput. Sliding-window chunking will be retained as a **baseline comparison strategy** for experimental evaluation.
